In [1]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("nudratabbas/software-developer-salary-prediction-dataset")
print(path)

c:\stuff\Code\Python\Machine-Learning\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


C:\Users\user\.cache\kagglehub\datasets\nudratabbas\software-developer-salary-prediction-dataset\versions\1


In [14]:
import os
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error as MSE, mean_absolute_error as MAE
from sklearn.model_selection import cross_val_score, train_test_split
from sklearn.preprocessing import MultiLabelBinarizer, StandardScaler
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor 

In [3]:
data = pd.read_csv(os.path.join(path, os.listdir(path)[1]))
data.head()

,experience,country,education,languages,frameworks,company_size,salary_usd
0,10,Singapore,Bachelors,"JavaScript, Go","Spring, Express",1-10,68491
1,13,USA,Some College,"PHP, Java","Flask, Ruby on Rails",51-200,120045
2,28,India,Some College,"JavaScript, C#","Vue, Spring",51-200,87811
3,16,Brazil,Bachelors,"Ruby, JavaScript","React, Spring",201-1000,99426
4,20,Singapore,PhD,"Java, Java","Laravel, Laravel",201-1000,108251


In [4]:
# print(len(data.select_dtypes(include="str").columns))

# for i in data.select_dtypes(include="object").columns:
#     print(i.upper())
#     print(f"Number of unique elements {data[i].nunique()}")    
#     print(f"unique elements: {data[i].unique()}\n",)    

# data["company_size"].unique()
data.info()
# data.isna().sum()

<class 'pandas.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 7 columns):
 #   Column        Non-Null Count  Dtype
---  ------        --------------  -----
 0   experience    10000 non-null  int64
 1   country       10000 non-null  str  
 2   education     10000 non-null  str  
 3   languages     10000 non-null  str  
 4   frameworks    10000 non-null  str  
 5   company_size  10000 non-null  str  
 6   salary_usd    10000 non-null  int64
dtypes: int64(2), str(5)
memory usage: 547.0 KB


In [5]:

mlb_lang=  MultiLabelBinarizer()
mlb_frame = MultiLabelBinarizer()

lang_encoded = pd.DataFrame(
    mlb_lang.fit_transform(data["languages"].str.split(", ")),
    columns=mlb_lang.classes_,
    index = data.index
)
frame_encoded = pd.DataFrame(
    mlb_frame.fit_transform(data["frameworks"].str.split(", ")),
                            columns=mlb_frame.classes_,
                            index = data.index
)

data.drop(columns=["languages","frameworks"], inplace=True)
data = pd.concat([data, lang_encoded, frame_encoded], axis=1)

size_order = {"1-10": 1, "11-50": 2 , "51-200": 3, "201-1000": 4, "1001-5000": 5, "5000+" : 6}
data["company_size"] = data["company_size"].map(size_order)

edu_order = {"High School": 1, "Some College": 2, "Bachelors": 3, "Masters": 4, "PhD": 5}
data["education"] = data["education"].map(edu_order)

data.head()

,experience,country,education,company_size,salary_usd,C#,C++,Go,Java,JavaScript,...,ASP.NET,Angular,Django,Express,Flask,Laravel,React,Ruby on Rails,Spring,Vue
0,10,Singapore,3,1,68491,0,0,1,0,1,...,0,0,0,1,0,0,0,0,1,0
1,13,USA,2,3,120045,0,0,0,1,0,...,0,0,0,0,1,0,0,1,0,0
2,28,India,2,3,87811,1,0,0,0,1,...,0,0,0,0,0,0,0,0,1,1
3,16,Brazil,3,4,99426,0,0,0,0,1,...,0,0,0,0,0,0,1,0,1,0
4,20,Singapore,5,4,108251,0,0,0,1,0,...,0,0,0,0,0,1,0,0,0,0


,experience,country,education,company_size,salary_usd,C#,C++,Go,Java,JavaScript,...,ASP.NET,Angular,Django,Express,Flask,Laravel,React,Ruby on Rails,Spring,Vue
0,10,Singapore,3,1,68491,0,0,1,0,1,...,0,0,0,1,0,0,0,0,1,0
1,13,USA,2,3,120045,0,0,0,1,0,...,0,0,0,0,1,0,0,1,0,0
2,28,India,2,3,87811,1,0,0,0,1,...,0,0,0,0,0,0,0,0,1,1
3,16,Brazil,3,4,99426,0,0,0,0,1,...,0,0,0,0,0,0,1,0,1,0
4,20,Singapore,5,4,108251,0,0,0,1,0,...,0,0,0,0,0,1,0,0,0,0


In [11]:
data_dummified = pd.get_dummies(data, columns=["country"])
# data_dummified.drop(columns=["languages", "frameworks"], inplace=True)
data_dummified.head()

,experience,education,company_size,salary_usd,C#,C++,Go,Java,JavaScript,PHP,...,country_Australia,country_Brazil,country_Canada,country_France,country_Germany,country_India,country_Japan,country_Singapore,country_UK,country_USA
0,10,3,1,68491,0,0,1,0,1,0,...,False,False,False,False,False,False,False,True,False,False
1,13,2,3,120045,0,0,0,1,0,1,...,False,False,False,False,False,False,False,False,False,True
2,28,2,3,87811,1,0,0,0,1,0,...,False,False,False,False,False,True,False,False,False,False
3,16,3,4,99426,0,0,0,0,1,0,...,False,True,False,False,False,False,False,False,False,False
4,20,5,4,108251,0,0,0,1,0,0,...,False,False,False,False,False,False,False,True,False,False


In [15]:
linreg = LinearRegression()

X = data_dummified.drop("salary_usd", axis=1).values
y = data_dummified["salary_usd"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

linreg.fit(X_train_scaled, y_train)
y_pred = linreg.predict(X_test_scaled)

score = linreg.score(X_test_scaled, y_test)
mse = MSE(y_test, y_pred)
root_MSE = np.sqrt(mse)
mae = MAE(y_test, y_pred)

train_score = linreg.score(X_train_scaled, y_train)
test_score = linreg.score(X_test_scaled, y_test)
print(f"Train R²: {train_score:.4f}")
print(f"Test R²:  {test_score:.4f}")

print(f"SCORE: {score}")
print(f"MSE: {mse}")
print(f"RMSE: {root_MSE}")
print(f"MAE: {mae}")


Train R²: 0.8725
Test R²:  0.8701
SCORE: 0.8701200358669585
MSE: 289688095.3040489
RMSE: 17020.22606501009
MAE: 13957.301278051322


Predictions are off by 14k According to the MAE